# Run all pipelines sequentially


In [ ]:
import sys, subprocess, time
from pathlib import Path

# This notebook must live in / be run from src/notebooks/.
NB_DIR = Path.cwd()
assert (NB_DIR / "pipeline_Helpdesk.ipynb").exists(), \
    f"run this notebook from src/notebooks/ (cwd={NB_DIR})"

KERNEL_NAME = "python3"   # registered kernel in this virtualenv

PIPELINES = [
    "pipeline_Helpdesk.ipynb",
    "pipeline_Sepsis.ipynb",
    "pipeline_BPIC20_DD.ipynb",
    "pipeline_Procurement.ipynb",
]
print("notebooks dir :", NB_DIR)
print("kernel        :", KERNEL_NAME)
print("pipelines     :", PIPELINES)

In [ ]:
# Run each pipeline as an isolated subprocess
results = []
overall_t0 = time.time()

for name in PIPELINES:
    path = NB_DIR / name
    if not path.exists():
        print(f"\u26a0 SKIP {name} (not found)", flush=True)
        results.append((name, "MISSING", 0.0))
        continue

    log = NB_DIR / f"{Path(name).stem}.run.log"
    print(f"\u25b6 START {name}   ({time.strftime('%Y-%m-%d %H:%M:%S')})   log -> {log.name}", flush=True)
    t0 = time.time()
    cmd = [sys.executable, "-m", "nbconvert", "--to", "notebook", "--execute", "--inplace",
           f"--ExecutePreprocessor.kernel_name={KERNEL_NAME}",
           "--ExecutePreprocessor.timeout=-1", str(path)]
    with open(log, "w") as lf:
        proc = subprocess.run(cmd, cwd=str(NB_DIR), stdout=lf, stderr=subprocess.STDOUT)
    dt = time.time() - t0
    status = "OK" if proc.returncode == 0 else f"FAILED (rc={proc.returncode})"
    results.append((name, status, dt))
    print(f"\u25c0 DONE  {name}: {status}   ({dt/60:.1f} min)   (details in {log.name})", flush=True)

print(f"\n{'='*72}\n=== SUMMARY  (total {(time.time()-overall_t0)/60:.1f} min) ===\n{'='*72}")
for name, status, dt in results:
    mark = "\u2713" if status == "OK" else "\u2717"
    print(f"  {mark} {name:30s} {status:18s} {dt/60:7.1f} min")
n_ok = sum(1 for _, s, _ in results if s == "OK")
print(f"\n{n_ok}/{len(results)} notebooks completed OK.")
if n_ok != len(results):
    print("Some notebooks did not finish cleanly \u2014 see the .run.log files above.")